# CODA TB V5 · BEATs + denyut + klinis lengkap
**Built with PriorLabs-TabPFN**

**EDA → audit → fitur jurnal / BEATs → nested CV → kalibrasi → evaluasi → XAI → RETURN**

V5 menguji nilai tambah representasi audio pralatih di luar seluruh **16 variabel klinis/demografi jurnal, termasuk denyut**. Model lengkap riset dan model mobile dilaporkan terpisah. Peningkatan terhadap baseline adalah hipotesis, bukan hasil yang dijanjikan.

**Semua tambahan library, bobot, dan kode encoder tersedia lokal dalam paket.** Sel pertama hanya memasang tambahan yang diperlukan dari `wheels_v5`, tanpa internet, tanpa mengganti PyTorch. Gunakan kernel V4 yang sudah berhasil: Windows x64, Python 3.11, torch 2.6.0+cu118. Mulai dari kernel yang baru/restart.

**Batas evaluasi:** 10×5 CV pasien independen dengan split lokal V4, kalibrasi 20%, seleksi inner UAR lalu AUC. Kode publik penulis menggunakan 15% dan StratifiedGroupKFold; V5 bukan reproduksi persis paper. Kohort 1.040 peserta telah dipakai pada versi sebelumnya; evaluasi tetap eksploratif. Denyut kamera belum tervalidasi. Penelitian skrining, bukan diagnosis.

## 0 · Ekstraksi paket dan setup lokal
Ekstrak **seluruh** `CODA_TB_V5_PACKAGE.zip` ke folder proyek yang sudah berisi data. Jangan menyalin notebook saja.

```text
D:/haqy/
  CODA_TB_V5_Research.ipynb
  research_package/
    config_v5.json
    assets_v5.json
    src/ (termasuk vendor_beats/)
    pretrained/v5/
    wheels_v5/
  dataset/dataset/{raw_data,meta_data}/
  dataset-audit/local-full/solicited_training_manifest_with_review_flags.csv
```

`MODE="full"` untuk PC CUDA. `MODE="preflight"` untuk audit/EDA tanpa melatih classifier TB. `RUN_NAME` yang sama melanjutkan checkpoint; perubahan kode/data/library/config memerlukan nama baru. Batas VRAM **45 GB desimal** dengan headroom 3 GB; pemakaian GPU lain tetap diperiksa. Memenuhi VRAM bukan sasaran kecepatan.

In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
for key in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS"]:os.environ.setdefault(key,"8")
MODE = "full"
RUN_NAME = "experiment_v5_001"
INSTALL_MISSING_LOCAL = True
PACKAGE_DIR = ""  # contoh r"<project>/research"
PROJECT_DIR = ""  # contoh r"<project>/data"
candidates=[Path(PACKAGE_DIR)] if PACKAGE_DIR else []
for parent in [Path.cwd(),*Path.cwd().parents]:
    candidates.extend([parent,parent/"research",parent/"research"])
PACKAGE=next((p.resolve() for p in candidates if (p/"config_v5.json").is_file() and (p/"src/coda_v5.py").is_file()),None)
if PACKAGE is None:
    raise RuntimeError(f"Paket V5 tidak ditemukan dari {Path.cwd()}. Ekstrak seluruh ZIP atau isi PACKAGE_DIR.")
sys.path.insert(0,str(PACKAGE))
from src.coda_v5_setup import setup
print(setup(PACKAGE,install=INSTALL_MISSING_LOCAL))
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
from src import coda_v5 as v5
from src import coda_v5_models as models
from src import coda_v5_features as features
from src import coda_v5_plots as plots
cfg=v5.configure(PACKAGE,RUN_NAME if MODE=="full" else RUN_NAME+"_preflight",MODE,PROJECT_DIR or PACKAGE.parent/"data")
OUT=Path(cfg["output_dir"])
display(pd.DataFrame(list(cfg["versions"].items()),columns=["Library","Versi"]))
print("Kernel:",sys.executable,"| Dataset:",cfg["data_root"],"| Output:",OUT)
print("CUDA:",torch.cuda.is_available(),"| Batas GPU total:",cfg["vram_limit_gb"],"GB (desimal)")

## 1 · Audit input, label, dan integritas audio
Semua hash WAV diverifikasi. Metadata sumber dibaca ulang. HR di luar 30–220 BPM menjadi missing sesuai kebijakan V1–V4. Negara, hasil lab, ID, dan jumlah batuk tidak digunakan sebagai prediktor. Semua subtipe riwayat TB dipertahankan sebagai kolom terpisah; tidak dipaksa eksklusif.

In [ ]:
participants,manifest=v5.audit_cohort(cfg)
clinical=features.encode_clinical(participants)
display(pd.DataFrame([{"Peserta":len(participants),"Batuk":len(manifest),"TB":int(participants.tb_status.sum()),
    "Non-TB":int((participants.tb_status==0).sum()),"HR missing":int(clinical.heart_rate.isna().sum())}]))
display(clinical[models.CLINICAL+["heart_rate"]].describe().T)
clinical[models.CLINICAL+["heart_rate"]].isna().sum().to_csv(OUT/"clinical_missingness.csv")
plots.show(plots.eda(participants,manifest,cfg))
plots.show(plots.audio_examples(manifest,cfg))

## 2 · Protokol dan daftar pembanding dibekukan
**44 konfigurasi tetap, 12 keluaran model + `selected_full`**; maksimum 2.200 evaluasi konfigurasi-inner. Enam kandidat kontrol menggunakan pooling pasien/head/PCA yang sama dengan BEATs untuk memisahkan efek representasi dari cara agregasi. Empat konfigurasi ensemble menggunakan ulang prediksi model dasar yang telah ditentukan, tanpa fit ulang di inner. Semua memilih mean inner **UAR**, tie-break AUC lalu ID, sebelum outer-test. `selected_full` adalah endpoint utama: memilih keluarga model BEATs/ensemble juga hanya dari inner CV, tanpa fit tambahan atau memilih skor outer-test tertinggi.

Keluaran: klinis lengkap saja; fitur audio jurnal + klinis lengkap (LR/CB/TabPFN); BEATs + klinis lengkap (LR/CB/TabPFN); ensemble terbatas; BEATs + denyut; BEATs + denyut + kuesioner mobile. Masing-masing melaporkan kandidat pemenangnya, bukan hanya skor terbaik keseluruhan.

Baseline audio jurnal memakai LibROSA dan rumus fitur kode penulis; pencarian hyperparameter dan split tetap implementasi lokal dengan anggaran terbatas, bukan reproduksi identik publikasi.

In [ ]:
splits=v5.nested_splits(participants,cfg)
reference=PACKAGE/"references/v5_v4/nested_splits.json"
if reference.exists():
    old=json.loads(reference.read_text())
    assert splits["folds"]==old["folds"],"Split berubah dari V4. Audit dahulu."
grid=models.candidate_grid(cfg)
display(pd.DataFrame(grid).groupby(["output","family"]).size().rename("Konfigurasi").reset_index())
print("Total kandidat:",len(grid),"| Evaluasi konfigurasi-inner:",len(grid)*cfg["outer_folds"]*cfg["inner_folds"])
print("Set klinis lengkap:",models.CLINICAL+["heart_rate"])

## 3 · Preprocessing dan ekstraksi fitur
**Baseline:** audio fisik, resample LibROSA 16 kHz, 261 fitur MFCC/chroma/spektral dan statistik sesuai rumus kode acuan. Undefined moments menjadi missing untuk imputasi training fold.

**BEATs:** encoder pralatih frozen, bukan fine-tuning TB; input amplitudo PCM fisik, 16 kHz, window 0,5 detik dengan padding kanan. Token di-mean menjadi 768 embedding per batuk, lalu mean/std batuk per peserta. Padding tidak menambah informasi. Encoder tidak menerima label. Ekstraksi dilakukan sekali, memiliki cache dan checkpoint per batch.

Batch GPU dipilih dari microbenchmark throughput dengan batas memori. PCA 32/64, imputasi, scaler, dan classifier **hanya fit di training fold**. Baseline LR/CB jurnal mengklasifikasi per batuk lalu mean probabilitas peserta. BEATs dan TabPFN memakai pooling peserta yang terdokumentasi.

Mode preflight menghitung fitur jurnal dan EDA saja; tidak membuat embedding BEATs seluruh kohort atau melatih classifier TB.

In [ ]:
table=features.prepare_features(participants,manifest,cfg,include_beats=(MODE=="full"))
display(table[["tb_status","heart_rate","temperature","height","weight"]].head())
plots.show(plots.preprocessing(table,splits,cfg))
print("Fitur tersimpan:",OUT/"participant_features.joblib")
provenance=json.loads((OUT/"feature_provenance.json").read_text())
if MODE=="full":display(pd.DataFrame(provenance.get("beats",{}).get("batch_benchmark",[])))
if MODE=="preflight":
    v5.atomic_json(OUT/"PREFLIGHT_COMPLETE.json",{"passed":True,"participants":len(table),"classifier_training":False,"full_beats_extraction":False})

## 4 · Nested CV dengan resume per inner fold
Model klasik di CPU; TabPFN di CUDA. Data kecil tidak selalu lebih cepat dengan CatBoost GPU. Tidak ada fine-tuning encoder atau attention search V4 yang mahal.

Isotonic dipelajari dari OOF proper-training kandidat terpilih, kemudian model di-refit pada proper-training. Cutoff Youden dan target sensitivitas 90% dipilih dari kalibrasi terpisah. Model/cutoff dibekukan sebelum evaluasi test. Target kalibrasi bukan jaminan sensitivitas test.

Resume memakai konfigurasi dan library identik. Model TabPFN disimpan sebagai konteks training CPU dan bobot lokal yang dirujuk eksplisit; tidak bergantung pada objek CUDA terserialisasi. Callback visual hanya untuk pelaporan, bukan pemilihan model.

In [ ]:
if MODE=="full":
    def on_fold(fold):plots.show(plots.fold_overview(fold,cfg))
    v5.run_nested_cv(table,splits,cfg,on_fold=on_fold)
else:
    print("PREFLIGHT: training classifier TB dilewati.")

## 5 · Evaluasi semua model: AUC, AP, FN/FP, kalibrasi
Endpoint utama mean ± sample SD antar-outer-fold, bukan pooled AUC saja. Pooled OOF merupakan hasil sekunder dari model fold berbeda. Confusion matrix menggunakan cutoff setiap fold, tidak menyesuaikan threshold dari label test. Laporkan hasil yang rendah juga.

In [ ]:
if MODE=="full":
    metrics,predictions,summary,pooled,paired=v5.aggregate_results(splits,cfg)
    display(summary[["combination","policy","roc_auc_mean","roc_auc_std","pr_auc_mean","sensitivity_mean","specificity_mean","brier_mean"]])
    display(pooled)
    plots.show(plots.evaluation(metrics,predictions,summary,pooled,paired,cfg))
    plots.show(plots.errors_calibration(predictions,cfg))
    plots.show(plots.diagnostics(predictions,paired,cfg))
    timing=pd.concat([pd.read_csv(OUT/f'outer_{f["outer_fold"]:02d}'/"timing.csv") for f in splits["folds"]])
    display(Markdown("**Waktu fit awal yang tercatat; resume memakai ulang checkpoint, bukan mengulang durasi ini.**"))
    display(timing.groupby("family").seconds.agg(["count","sum","mean"]))

## 6 · Uji kontribusi, bukan sekadar skor tertinggi
Perbandingan utama: BEATs vs fitur jurnal dalam keluarga classifier yang sama; multimodal vs klinis lengkap saja; ensemble vs komponennya; input mobile lengkap vs audio+denyut.

CI bootstrap OOF bersifat kondisional pada prediksi model terlatih: tanpa retraining uncertainty atau koreksi banyak perbandingan. Kenaikan hanya karena menambah klinis bukan bukti inovasi audio/model. AUC Sensors sekitar 0,81 adalah konteks dari kohort berbeda. Nilai di atas 0,81 tidak otomatis membuktikan superioritas/novelty.

Jangan menggunakan hasil outer-test untuk mengubah grid, memilih seed, atau mengulang sampai mendapat angka tinggi. Cohort reuse V1–V5 tetap membatasi klaim.

In [ ]:
if MODE=="full":
    display(paired)
    baseline=PACKAGE/"references/v5_v4/nested_mean_std.csv"
    if baseline.exists():
        old=pd.read_csv(baseline)
        display(Markdown("**Konteks V4: input klinis dan aturan seleksi berbeda; bukan ablation terkontrol V5.**"))
        display(old.loc[old.policy=="youden",["combination","roc_auc_mean","roc_auc_std"]])

## 7 · XAI, simulasi error denyut, dan cascade mobile
Seluruh training/evaluasi sudah tersimpan sebelum bagian ini. Kegagalan XAI tidak menyebabkan training ulang; rerun sel ini melanjutkan postprocessing per outer fold.

Full output untuk XAI dipilih dari inner CV, bukan test. Exact grouped Shapley menjelaskan audio/klinis/denyut pada empat peserta acak per fold; individual Monte Carlo Shapley menjelaskan dua peserta per fold. Background dari proper-training, permutation seed tetap pada dua background. Completeness dan perbedaan batch/single dicatat; bukan bukti kausal/validasi klinis.

Cascade memakai **BEATs + denyut → BEATs + denyut + kuesioner mobile**, bukan model riset yang memerlukan tinggi/berat/temperatur dengan input diam-diam dikosongkan. HR missing memaksa tahap lanjut, tetapi imputasi tetap keterbatasan; aplikasi harus meminta pengukuran ulang jika kualitasnya buruk. Gate target 99% dan cutoff akhir target 90% dipilih pada kalibrasi. Laporkan proporsi kuesioner dan TB yang berhenti terlalu awal.

HR ±5/±10 BPM adalah simulasi, bukan validasi kamera. Audio yang sama digunakan kembali di tahap lanjutan tanpa ekstraksi embedding kedua.

In [ ]:
if MODE=="full":
    v5.finish_postprocessing(table,splits,cfg)
    display(pd.read_csv(OUT/"cascade_pooled_counts.csv"))
    display(pd.read_csv(OUT/"nested_individual_xai.csv").head(20))
    plots.show(plots.postprocessing(cfg))

## 8 · Ekspor hasil untuk audit di laptop
RETURN memuat konteks/model, fitur peserta, prediksi, cutoff, split, log pencarian, XAI, robustness, cascade, gambar, source, dan checksum. Bobot pralatih tidak diduplikasi dalam RETURN; laptop sudah memilikinya dari paket V5.

Belum ada model deployment final. Jangan memilih outer fold dengan skor tertinggi untuk aplikasi. Setelah audit, pelatihan model deployment dan evaluasi independen merupakan pekerjaan terpisah.

In [ ]:
if MODE=="full":
    returned=v5.export_return(cfg,splits)
    print("Selesai. Salin kembali ke laptop:",returned)
else:
    print("Preflight selesai. Untuk training: pindahkan seluruh paket V5, gunakan kernel V4, MODE=full, lalu Run All.")

## Sumber dan batas klaim
- [Sensors 2026](https://www.mdpi.com/1424-8220/26/4/1223) · [kode penulis](https://github.com/gpk1983/Reproducible-TB-Screening-from-Cough-Audio)
- [BEATs](https://arxiv.org/abs/2212.09058) · [implementasi resmi Microsoft](https://github.com/microsoft/unilm/tree/master/beats)
- [TabPFN v2, Nature 2025](https://www.nature.com/articles/s41586-024-08328-6) · [bobot dan lisensi v2](https://huggingface.co/Prior-Labs/TabPFN-v2-clf)

Kode BEATs berasal dari upstream Microsoft dengan perubahan relative import saja. Bobot BEATs memakai mirror pihak ketiga karena unduhan penerbit tidak tersedia; hash sama pada dua mirror, tetapi tidak diautentikasi terhadap unduhan penerbit. Provenance lengkap ada di `assets_v5.json`. TabPFN diambil dari repositori bobot resmi. Tidak ada download otomatis saat training.

Menggunakan model yang ada bukan klaim metode baru. Semua peningkatan, nilai tambah audio, manfaat cascade, dan kestabilan XAI harus diperiksa dari hasil. Akurasi kamera, generalisasi Indonesia, dan kemampuan diagnosis belum dibuktikan.